# Processament dels resultats d'AlphaFold3 - BBF-14

Aquest notebook recol·lecta i organitza els resultats de les prediccions estructurals d'AlphaFold3 per als 14 complexos del dataset BBF-14 i prepara les estructures per als càlculs posteriors de Rosetta.

La funció collectAlphaFold3Results de la classe sequenceModels recorre la carpeta af3_jobs_bbf14, llegeix els fitxers de sortida de cada complex i extreu les mètriques de confiança: ranking_score, summary_iptm, summary_ptm, summary_ranking_score, summary_fraction_disordered, summary_has_clash i les mètriques de cadena (summary_chain_iptm, summary_chain_pair_iptm, summary_chain_pair_pae_min, summary_chain_ptm). Per a cada complex es generen 5 estructures; la funció retorna tant el dataframe complet com el subset amb el millor model per complex (seleccionat per ranking_score màxim). Els resultats es guarden en format CSV a la carpeta af3_tables.

In [2]:
import sys
from pathlib import Path
from prepare_proteins import sequenceModels

base = Path.cwd().parent  # La meva base és la carpeta "projects"
sys.path.append(str(base))

fasta_file = base / "inputs" / "input_af3_bbf14.fasta"
af_folder = base / "Scripts" / "af3_jobs_bbf14"

# Carpeta on es guardaràn els fitxers .csv amb els resultats de ranking_scores
csv_folder = Path("./af3_tables")
csv_folder.mkdir(parents=True, exist_ok=True)

# Carpeta on es guardaran les estructures PDB
pdb_folder = Path("./af3_pdbs")
pdb_folder.mkdir(parents=True, exist_ok=True)

sequence_models = sequenceModels(str(fasta_file))

df_all_scores, df_selected_scores, copied_paths = sequence_models.collectAlphaFold3Results(
    af_folder=str(af_folder),
    return_selected=True,
    output_folder=str(pdb_folder),
    overwrite=True
)

print("\n=== TOTS ELS RESULTATS ===")
print(df_all_scores.head())

print("\n=== MILLOR PER MODEL ===")
print(df_selected_scores.head())

print("\nDimensions:")
print("scores_df:", df_all_scores.shape)
print("best_df:", df_selected_scores.shape)

print("\n=== PDBs GENERATS ===")
for model, files in copied_paths.items():
    print(model, "->", files)

/opt/anaconda3/envs/prepare_proteins/lib/python3.10/site-packages/prepare_proteins-0.0.1-py3.10.egg/prepare_proteins/_prepare_sequences.py:160: UserWarning: Sequences contain non-standard amino acid codes — Complex1: :, Complex2: :, Complex3: :, Complex4: :, Complex5: :, Complex6: :, Complex7: :, Complex8: :, Complex9: :, Complex10: :, Complex11: :, Complex12: :, Complex13: :, Complex14: :
  warnings.warn(



=== TOTS ELS RESULTATS ===
                      ranking_score  ... summary_ranking_score
model    seed sample                 ...                      
Complex2 1    0            0.753891  ...                  0.75
              1            0.759062  ...                  0.76
              2            0.762274  ...                  0.76
              3            0.756485  ...                  0.76
              4            0.751654  ...                  0.75

[5 rows x 10 columns]

=== MILLOR PER MODEL ===
                       ranking_score  ... summary_ranking_score
model     seed sample                 ...                      
Complex1  1    3            0.857156  ...                  0.86
Complex10 1    1            0.419977  ...                  0.42
Complex11 1    4            0.675917  ...                  0.68
Complex12 1    1            0.573140  ...                  0.57
Complex13 1    4            0.658324  ...                  0.66

[5 rows x 10 columns]

Dimensions

In [4]:
df_all_scores.to_csv(csv_folder / "af3_scores_all.csv")
df_selected_scores.to_csv(csv_folder / "af3_scores_best.csv")

best_sorted = df_selected_scores.sort_values("ranking_score", ascending=False)
best_sorted.to_csv(csv_folder / "af3_scores_best_sorted.csv")

print("\n=== MILLORS MODELS ORDENATS PER ranking_score ===")
print(best_sorted)

print("\nCSV guardats a:", csv_folder)


=== MILLORS MODELS ORDENATS PER ranking_score ===
                       ranking_score  ... summary_ranking_score
model     seed sample                 ...                      
Complex7  1    1            0.879330  ...                  0.88
Complex1  1    3            0.857156  ...                  0.86
Complex6  1    4            0.842577  ...                  0.84
Complex8  1    2            0.840230  ...                  0.84
Complex3  1    2            0.813006  ...                  0.81
Complex2  1    2            0.762274  ...                  0.76
Complex9  1    2            0.751718  ...                  0.75
Complex11 1    4            0.675917  ...                  0.68
Complex13 1    4            0.658324  ...                  0.66
Complex12 1    1            0.573140  ...                  0.57
Complex10 1    1            0.419977  ...                  0.42
Complex4  1    0            0.360429  ...                  0.36
Complex14 1    4            0.339899  ...            

## Retall dels extrems terminals de baixa confiança

Abans de procedir amb els càlculs de Rosetta, s'identifiquen i s'eliminen les regions terminals amb baixa confiança estructural mitjançant la funció removeTerminiByConfidenceScore de la classe proteinModels.

Es parteix dels fitxers .pdb obtinguts d'AlphaFold3 (carpeta af3_pdbs) i s'aplica el retall sobre la cadena A (target), que és la que pot presentar extrems desordenats. La cadena B (binder) no es modifica. Els PDBs retallats es guarden a pdbs_trim.

Un cop completat el procés, es va decidir no utilitzar les estructures retallades i continuar el pipeline amb els PDBs originals. Les cel·les següents documenten el retall realitzat, però les estructures de pdbs_trim no s'han utilitzat en l'anàlisi posterior.

In [1]:
from pathlib import Path
import shutil
from prepare_proteins import proteinModels
from Bio.PDB import PDBIO

input_pdbs_dir = Path("af3_pdbs")  # estàs a Scripts
output_dir = Path("../Inputs/pdbs_trim")
temp_dir = Path("temp_trim")

output_dir.mkdir(parents=True, exist_ok=True)
temp_dir.mkdir(exist_ok=True)

pdb_files = list(input_pdbs_dir.glob("*.pdb"))

print(f"Trobats {len(pdb_files)} PDBs")

for pdb_file in pdb_files:

    # carpeta temporal per aquest model
    model_temp_dir = temp_dir / pdb_file.stem
    if model_temp_dir.exists():
        shutil.rmtree(model_temp_dir)
    model_temp_dir.mkdir()

    # copiar el pdb dins la carpeta temporal
    copied_pdb = model_temp_dir / pdb_file.name
    shutil.copy2(pdb_file, copied_pdb)

    # carregar amb proteinModels (necessita carpeta)
    models = proteinModels(str(model_temp_dir))

    # aplicar trimming
    models.removeTerminiByConfidenceScore(
        confidence_threshold=70.0,
        keep_up_to=5,
        renumber=False,
        verbose=False,
        output=None  # IMPORTANT: evitar bug
    )

    # guardar resultat
    model_name = models.models_names[0]
    io = PDBIO()
    io.set_structure(models.structures[model_name])

    output_pdb = output_dir / pdb_file.name
    io.save(str(output_pdb))

print("\n Tots els PDBs han estat processats")

Trobats 14 PDBs

 Tots els PDBs han estat processats


In [1]:
from pathlib import Path
from Bio.PDB import PDBParser
import pandas as pd

orig_dir = Path("af3_pdbs")
trim_dir = Path("../Inputs/pdbs_trim")

parser = PDBParser(QUIET=True)

def get_chain_residue_ids(pdb_file, chain_id):
    structure = parser.get_structure(pdb_file.stem, str(pdb_file))
    model = next(structure.get_models())
    if chain_id not in model:
        return []
    residues = [r.id[1] for r in model[chain_id].get_residues() if r.id[0] == " "]
    return residues


def summarize_trim(orig_pdb, trim_pdb, chain_id):
    orig_ids = get_chain_residue_ids(orig_pdb, chain_id)
    trim_ids = get_chain_residue_ids(trim_pdb, chain_id)

    orig_len = len(orig_ids)
    trim_len = len(trim_ids)

    if not orig_ids or not trim_ids:
        return {
            f"{chain_id}_orig": orig_len,
            f"{chain_id}_trim": trim_len,
            f"{chain_id}_trim_N": None,
            f"{chain_id}_trim_C": None,
        }

    trim_n = orig_ids.index(trim_ids[0])
    trim_c = len(orig_ids) - orig_ids.index(trim_ids[-1]) - 1

    return {
        f"{chain_id}_orig": orig_len,
        f"{chain_id}_trim": trim_len,
        f"{chain_id}_trim_N": trim_n,
        f"{chain_id}_trim_C": trim_c,
    }

rows = []

for orig_pdb in sorted(orig_dir.glob("*.pdb")):
    trim_pdb = trim_dir / orig_pdb.name

    row = {"complex": orig_pdb.stem}
    row.update(summarize_trim(orig_pdb, trim_pdb, "A"))
    row.update(summarize_trim(orig_pdb, trim_pdb, "B"))
    rows.append(row)

df = pd.DataFrame(rows).sort_values("complex")
print(df.to_string(index=False))


  complex  A_orig  A_trim  A_trim_N  A_trim_C  B_orig  B_trim  B_trim_N  B_trim_C
 Complex1     122     110         0        12      85      85         0         0
Complex10     122     102         8        12      70      60        10         0
Complex11     122     110         0        12     178     178         0         0
Complex12     122     109         0        13     188     188         0         0
Complex13     122     109         0        13     127     127         0         0
Complex14     122     109         0        13     107     107         0         0
 Complex2     122     110         0        12     117     117         0         0
 Complex3     122     111         0        11     102      95         7         0
 Complex4     122      96        13        13     145     145         0         0
 Complex5     122      96        13        13     157     151         6         0
 Complex6     122     110         0        12     130     130         0         0
 Complex7     12

La validació del retall s'ha fet de manera observacional amb ChimeraX, sobreposant per a cada complex l'estructura original i la retallada per comprovar visualment quina regió s'ha eliminat. En la majoria de casos el retall afecta únicament la cadena A (target), amb reduccions que van des de pocs residus fins a trams més marcats als complexos 4 i 5. A la cadena B (binder) no s'observa cap modificació en cap dels complexos.

In [24]:
from pathlib import Path
from Bio.PDB import PDBParser
import pandas as pd
import numpy as np

orig_dir = Path("af3_pdbs")
trim_dir = Path("../Inputs/pdbs_trim")

parser = PDBParser(QUIET=True)

def get_chain_plddt(pdb_file, chain_id):
    structure = parser.get_structure(pdb_file.stem, str(pdb_file))
    model = next(structure.get_models())

    if chain_id not in model:
        return None

    chain = model[chain_id]
    residue_scores = []

    for res in chain:
        if res.id[0] != " ":
            continue

        atom_b = [a.bfactor for a in res.get_atoms()]
        if atom_b:
            residue_scores.append(np.mean(atom_b))

    if residue_scores:
        return np.mean(residue_scores)
    else:
        return None


rows = []

for orig_pdb in sorted(orig_dir.glob("*.pdb")):
    trim_pdb = trim_dir / orig_pdb.name

    A_orig = get_chain_plddt(orig_pdb, "A")
    B_orig = get_chain_plddt(orig_pdb, "B")
    A_trim = get_chain_plddt(trim_pdb, "A")
    B_trim = get_chain_plddt(trim_pdb, "B")

    rows.append({
        "complex": orig_pdb.stem,
        "A_plddt_orig": round(A_orig, 2) if A_orig else None,
        "A_plddt_trim": round(A_trim, 2) if A_trim else None,
        "A_delta": round(A_trim - A_orig, 2) if A_orig and A_trim else None,
        "B_plddt_orig": round(B_orig, 2) if B_orig else None,
        "B_plddt_trim": round(B_trim, 2) if B_trim else None,
        "B_delta": round(B_trim - B_orig, 2) if B_orig and B_trim else None,
    })

df = pd.DataFrame(rows).sort_values("complex")

print(df.to_string(index=False))

  complex  A_plddt_orig  A_plddt_trim  A_delta  B_plddt_orig  B_plddt_trim  B_delta
 Complex1         81.96         85.85     3.89         88.61         88.61     0.00
Complex10         73.72         78.34     4.62         69.18         72.79     3.61
Complex11         78.03         82.36     4.33         87.57         87.57     0.00
Complex12         73.80         78.07     4.27         87.65         87.65     0.00
Complex13         78.52         82.57     4.05         87.90         87.90     0.00
Complex14         77.40         82.24     4.85         83.70         83.70     0.00
 Complex2         81.32         85.37     4.05         84.50         84.50     0.00
 Complex3         83.10         87.14     4.04         83.66         85.93     2.27
 Complex4         69.72         76.39     6.67         87.19         87.19     0.00
 Complex5         68.90         74.75     5.85         75.99         76.94     0.94
 Complex6         82.50         86.68     4.17         87.14         87.14  

S'ha analitzat l'efecte del trimming sobre la qualitat estructural a partir dels valors de pLDDT (extrets dels B-factors) abans i després del procés. En tots els complexos la cadena A (target) presenta un augment del pLDDT mitjà després del retall, amb increments d'aproximadament +4 a +6 punts, fet que indica que l'eliminació de residus terminals de baixa confiança millora la qualitat global de l'estructura. La cadena B (binder) es manté pràcticament invariable en la majoria de casos, confirmant que no ha estat modificada. Tot i aquests resultats, es va decidir no incorporar les estructures retallades al pipeline i continuar amb els PDBs originals, de manera consistent amb el procediment aplicat al dataset IL-7Rα.